# 长期记忆演示（精简版）：`chat_with_memories`

本 notebook 在 `memory_text_vector_detail.ipynb` 的实现之上，用 **`chat_with_memories` 三步走** 串起检索 → 回答 → 写回记忆。

| 步骤 | 做什么 | 对应调用 |
|------|--------|----------|
| **Step 1** | 混合检索用户偏好记忆（向量 + BM25 + 融合） | `retriever.dense_search` / `bm25_search` / `fuse_search_results` |
| **Step 2** | 结合召回记忆回答用户问题 | `build_memory_prompt` + `assistant_llm.chat` |
| **Step 3** | 从本轮交互抽取新记忆并写入向量库 | `extractor.extract` → `memory_store.add` |

向量默认走 **阿里云 DashScope**（`.env` 里配 `DASHSCOPE_API_KEY`）。

## 0. 环境

```bash
pip install langchain-core langchain-openai chromadb rank-bm25 jieba openai python-dotenv httpx numpy
```

在项目根目录或 `Chapter-3` 上级目录放 `.env`。


In [1]:
import sys
from pathlib import Path

from memory_notebook_env import load_detail_notebook_defs, setup_notebook_env

if str(Path.cwd()) not in sys.path and (Path.cwd() / "memory_notebook_env.py").exists():
    sys.path.insert(0, str(Path.cwd()))
elif str(Path.cwd() / "Chapter-3") not in sys.path:
    chapter3_candidate = Path.cwd() / "Chapter-3"
    if (chapter3_candidate / "memory_notebook_env.py").exists():
        sys.path.insert(0, str(chapter3_candidate))

env = setup_notebook_env()
CHAPTER3 = env.chapter3
ROOT = env.root

Chapter-3: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-3
项目根目录: D:\myproject\mira-ai-lab\agent-systems-book
DASHSCOPE_API_KEY: 已配置


## 1. 加载基础实现

从 **`memory_hybrid_compressor.py`**（源码 A）与 **`memory_ltm_improved.py`**（源码 B）直接 import，定义 `HybridRetriever`、`MemoryCompressor`、`SelfBuiltLongTermMemoryImproved` 等。

In [2]:
load_detail_notebook_defs(globals(), env=env)

C:\Users\zhanghong26\.conda\envs\agent-systems-book\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


✓ 已从 memory_hybrid_compressor / memory_ltm_improved 加载定义


11

## 2. 封装组件 + `chat_with_memories`

实现见 **`memory_chat.py`**（`MemoryHybridRetriever`、`MemoryStore`、`chat_with_memories` 等）。

In [4]:
from types import SimpleNamespace

from memory_chat import (
    DEMO_USER_ID,
    AssistantLLM,
    MemoryExtractor,
    MemoryHybridRetriever,
    MemoryStore,
    chat_with_memories,
)

USER_ID = DEMO_USER_ID
print("✓ 已从 memory_chat.py 加载封装组件")

✓ 已从 memory_chat.py 加载封装组件


## 3. 初始化记忆库与组件

In [5]:
CHROMA_DIR = CHAPTER3 / "chroma_notebook_demo_d"

llm = default_llm()
ltm = get_or_reset_ltm(
    user_id=USER_ID,
    persist_directory=CHROMA_DIR,
    embedding_model="dashscope",
    collection_name="notebook_long_term_d_v1",
    similarity_threshold=0.0,
)

retriever = MemoryHybridRetriever(ltm)
memory_store = MemoryStore(ltm)
extractor = MemoryExtractor(MemoryCompressor(llm))
assistant_llm = AssistantLLM(llm)

# 预置两条样例记忆
if not ltm.documents:
    memory_store.add(
        SimpleNamespace(
            data={
                "content": "用户喜欢住海景房，预算每晚不超过800元",
                "summary": "用户偏好海景房，预算每晚不超过800元",
                "key_points": ["喜欢海景房", "预算每晚不超过800元"],
                "importance": 0.9,
                "metadata": {"memory_type": "preference"},
            },
            metadata={"key_points": ["喜欢海景房", "预算每晚不超过800元"]},
        ),
        user_id=USER_ID,
    )
    memory_store.add(
        SimpleNamespace(
            data={
                "content": "用户计划下个月去三亚旅游",
                "summary": "用户计划下个月去三亚旅游",
                "key_points": ["下个月去三亚"],
                "importance": 0.5,
                "metadata": {"memory_type": "fact"},
            },
            metadata={"key_points": ["下个月去三亚"]},
        ),
        user_id=USER_ID,
    )

print(f"库内记忆条数: {len(ltm.documents)}")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ZHANGH~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.365 seconds.
Prefix dict has been built successfully.


库内记忆条数: 2


## 4. 运行 `chat_with_memories`

In [6]:
RUN_LLM = True  # 改为 False 可只看检索结果，不调大模型

message = "你还记得我喜欢住什么样的酒店吗？"

if RUN_LLM:
    result = chat_with_memories(
        message,
        retriever=retriever,
        memory_store=memory_store,
        extractor=extractor,
        assistant_llm=assistant_llm,
        user_id=USER_ID,
    )
    print("【召回记忆】")
    for h in result["relevant_memories"]:
        print(f"  - {h.get('summary') or h.get('content')} (score={h.get('final_score', 0):.4f})")
    print("\n【助手回答】")
    print(result["assistant_response"])
    print("\n【新抽取要点】", result["extracted_key_points"])
    print("【写入结果】", result["memory_write_result"])
else:
    user_records = memory_store.get_user_records(USER_ID)
    dense_result = retriever.dense_search(message, USER_ID, 6)
    bm25_hits = retriever.bm25_search(message, user_records, 6)
    hits = retriever.fuse_search_results(
        query=message,
        records=user_records,
        dense_ids=dense_result["ids"],
        dense_distances=dense_result["distances"],
        bm25_hits=bm25_hits,
        top_k=3,
        threshold=0.0,
    )
    print("【Step 1 召回】")
    for h in hits:
        print(f"  - {h.get('summary') or h.get('content')}")
    print("\nRUN_LLM=False：跳 Step 2/3。改 True 跑完整 chat_with_memories。")

【召回记忆】
  - 用户偏好海景房，预算每晚不超过800元 (score=0.0130)
  - 用户计划下个月去三亚旅游 (score=0.0122)

【助手回答】
当然记得！根据您之前的偏好，您喜欢入住海景房，并且希望每晚的预算控制在 800 元以内。您下个月还计划去三亚旅游，这些信息我已经记录在案。如果您需要我帮您挑选符合这些条件的酒店，随时告诉我！祝您旅途愉快～

【新抽取要点】 ['喜欢海景房', '每晚预算≤800元', '下个月计划去三亚旅游']
【写入结果】 {'ids': ['13cc2a0c-a958-48ee-9f6c-3d68cc3f2784'], 'key_points': ['喜欢海景房', '每晚预算≤800元', '下个月计划去三亚旅游']}


## 小结

`chat_with_memories`（见 `memory_chat.py`）把长期记忆闭环串成三步：

1. **检索**：dense + BM25 + 融合
2. **使用**：记忆拼进 Prompt，调用 LLM 回答
3. **写入**：从本轮对话抽取要点，入库（含去重/合并）

底层实现见 `memory_hybrid_compressor.py`、`memory_ltm_improved.py`；完整分步讲解见 `memory_text_vector_detail.ipynb`。